In [43]:
import os
import json
import re
import glob
import pandas as pd

def extract_rows_from_slurm_ids(slurm_ids):
    data = []

    for slurm_id in slurm_ids:
        try:
            log_path = f"/n/home08/atong/projects/LMVF/logs/output_{slurm_id}.log"
            if not os.path.isfile(log_path):
                raise FileNotFoundError(f"Log file not found for slurm_id {slurm_id}")

            with open(log_path) as f:
                log_content = f.read()

            # Extract output directory from the log
            output_match = re.search(r"Output directory:\s+(.*)", log_content)
            if not output_match:
                raise ValueError(f"Output directory not found in log for slurm_id {slurm_id}")

            path = output_match.group(1).strip()
            cfg_path = os.path.join(path, "cfg")

            # Load config.json
            with open(os.path.join(cfg_path, "config.json")) as f:
                config = json.load(f)

            # Load max_new_tokens from model configs
            def get_max_new_tokens(filename):
                with open(os.path.join(cfg_path, filename)) as f:
                    return json.load(f)["model"]["max_new_tokens"]

            row = {
                "model": config["model"],
                "verifier_model": config["verifier_model"],
                "strict_verifier_model": config["strict_verifier_model"],
                "ntasks": config["gpus_per_node"]*config["num_nodes"],
                "dataset": config["dataset"],
                "num_generations": config["num_generations"],
                "max_new_tokens_model": get_max_new_tokens("model_config.json"),
                "max_new_tokens_verifier": get_max_new_tokens("verifier_model_config.json"),
                "max_new_tokens_strict_verifier": get_max_new_tokens("strict_verifier_model_config.json"),
                "slurm_id": slurm_id,
                "path": path
            }

            # Aggregate accuracy_task_*.json
            accuracy_files = glob.glob(os.path.join(path, "accuracy_task_*.json"))
            if not accuracy_files:
                raise ValueError("No accuracy_task_*.json files found.")

            metric_sets = []
            acc_data_all = []
            for file in accuracy_files:
                with open(file) as f:
                    data_json = json.load(f)
                    acc_data_all.append(data_json)
                    metric_sets.append(set(data_json.keys()))

            if not all(metrics == metric_sets[0] for metrics in metric_sets):
                raise ValueError(f"Mismatch in accuracy metrics among files in {path}")

            metrics = metric_sets[0]
            for metric in metrics:
                total_correct = sum(d[metric]["correct_count"] for d in acc_data_all)
                total_eval = sum(d[metric]["total_eval"] for d in acc_data_all)
                accuracy = total_correct / total_eval if total_eval > 0 else None

                safe_metric = metric.replace("-", "").replace("@", "")
                row[f"{safe_metric}_correct_count"] = total_correct
                row[f"{safe_metric}_total_eval"] = total_eval
                row[f"{safe_metric}_accuracy"] = accuracy

            # Execution time
            time_matches = re.findall(r"Total execution time: (\d+):(\d+):(\d+)", log_content)
            times_seconds = [
                int(h)*3600 + int(m)*60 + int(s)
                for h, m, s in time_matches
            ]

            if times_seconds:
                total_seconds = sum(times_seconds)
                row["total_exec_time_minutes"] = total_seconds / 60
            else:
                row["total_exec_time_minutes"] = None

            data.append(row)

        except Exception as e:
            print(f"Error processing slurm_id {slurm_id}\n{e}")
            continue

    return pd.DataFrame(data)


In [46]:
# 11486404
slurm_ids = ['10976441', '11066453', '11104922', '11109136', '11124341', '11286669', '11324997', '11395474', '11406423', '11459919', '11526101', '11527839', '11528328', '11534401', '11535609']
rows = extract_rows_from_slurm_ids(slurm_ids)

In [47]:
rows

,model,verifier_model,strict_verifier_model,ntasks,dataset,num_generations,max_new_tokens_model,max_new_tokens_verifier,max_new_tokens_strict_verifier,slurm_id,...,pass8_correct_count,pass8_total_eval,pass8_accuracy,pass1_correct_count,pass1_total_eval,pass1_accuracy,total_exec_time_minutes,pass4_correct_count,pass4_total_eval,pass4_accuracy
0,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,2,math,8,2048,32768,16,10976441,...,318.0,500.0,0.636,214,500,0.428,298.466667,NaN,NaN,NaN
1,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,2,math,8,2048,8192,16,11066453,...,314.0,500.0,0.628,200,500,0.400,155.166667,NaN,NaN,NaN
2,gemma-3-4b-it,gemma-3-4b-it,gemma-3-4b-it,2,math,8,2048,2048,16,11104922,...,416.0,500.0,0.832,347,500,0.694,49.450000,NaN,NaN,NaN
3,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,deepseek-r1-distill-qwen-1.5b,2,math,8,2048,8192,2048,11109136,...,315.0,500.0,0.630,186,500,0.372,172.066667,NaN,NaN,NaN
4,deepseek-r1-distill-qwen-1.5b,deepseek-r1-distill-qwen-1.5b,deepseek-r1-distill-qwen-1.5b,2,math,8,8192,8192,2048,11124341,...,445.0,500.0,0.890,356,500,0.712,152.633333,NaN,NaN,NaN
5,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-1b-it,2,math,8,2048,8192,16,11286669,...,320.0,500.0,0.640,193,500,0.386,136.266667,NaN,NaN,NaN
6,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,2,math,8,2048,4096,16,11324997,...,317.0,500.0,0.634,198,500,0.396,103.866667,NaN,NaN,NaN
7,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,2,math,8,2048,2048,16,11395474,...,309.0,500.0,0.618,197,500,0.394,71.316667,NaN,NaN,NaN
8,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,2,math,4,2048,4096,16,11406423,...,NaN,NaN,NaN,199,500,0.398,59.716667,280.0,500.0,0.560
9,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,2,math,4,1024,2048,16,11459919,...,NaN,NaN,NaN,187,500,0.374,39.516667,255.0,500.0,0.510


In [48]:
selected_columns = [
    'model',
    'verifier_model',
    'strict_verifier_model',
    'max_new_tokens_model',
    'max_new_tokens_verifier',
    'max_new_tokens_strict_verifier',
    'pass1_accuracy',
    'bonmav_accuracy',
    'pass4_accuracy',
    'pass8_accuracy',
    'total_exec_time_minutes'
]

filtered_df = rows[selected_columns]

In [49]:
filtered_df

,model,verifier_model,strict_verifier_model,max_new_tokens_model,max_new_tokens_verifier,max_new_tokens_strict_verifier,pass1_accuracy,bonmav_accuracy,pass4_accuracy,pass8_accuracy,total_exec_time_minutes
0,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,2048,32768,16,0.428,0.572,NaN,0.636,298.466667
1,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,2048,8192,16,0.400,0.554,NaN,0.628,155.166667
2,gemma-3-4b-it,gemma-3-4b-it,gemma-3-4b-it,2048,2048,16,0.694,0.724,NaN,0.832,49.450000
3,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,deepseek-r1-distill-qwen-1.5b,2048,8192,2048,0.372,0.484,NaN,0.630,172.066667
4,deepseek-r1-distill-qwen-1.5b,deepseek-r1-distill-qwen-1.5b,deepseek-r1-distill-qwen-1.5b,8192,8192,2048,0.712,0.790,NaN,0.890,152.633333
5,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-1b-it,2048,8192,16,0.386,0.450,NaN,0.640,136.266667
6,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,2048,4096,16,0.396,0.552,NaN,0.634,103.866667
7,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,2048,2048,16,0.394,0.536,NaN,0.618,71.316667
8,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,2048,4096,16,0.398,0.510,0.560,NaN,59.716667
9,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,1024,2048,16,0.374,0.460,0.510,NaN,39.516667


In [50]:
self_verifier_filtered_df = filtered_df.iloc[[11,2,4]]

In [51]:
selected_columns_2 = [
    'model',
    'verifier_model',
    'strict_verifier_model',
    'max_new_tokens_model',
    'max_new_tokens_strict_verifier',
    'pass1_accuracy',
    'bonmav_accuracy',
    'pass8_accuracy',
    'total_exec_time_minutes'
]
self_verifier_filtered_df[selected_columns_2]

,model,verifier_model,strict_verifier_model,max_new_tokens_model,max_new_tokens_strict_verifier,pass1_accuracy,bonmav_accuracy,pass8_accuracy,total_exec_time_minutes
11,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,1024,16,0.322,0.460,NaN,31.833333
2,gemma-3-4b-it,gemma-3-4b-it,gemma-3-4b-it,2048,16,0.694,0.724,0.832,49.450000
4,deepseek-r1-distill-qwen-1.5b,deepseek-r1-distill-qwen-1.5b,deepseek-r1-distill-qwen-1.5b,8192,2048,0.712,0.790,0.890,152.633333


In [52]:
diff_strict_filtered_df = filtered_df.iloc[[1,3,5]]

In [53]:
selected_columns_3 = [
    'model',
    'verifier_model',
    'strict_verifier_model',
    'max_new_tokens_strict_verifier',
    'pass1_accuracy',
    'bonmav_accuracy',
    'pass8_accuracy',
    'total_exec_time_minutes'
]

In [54]:
diff_strict_filtered_df[selected_columns_3]

,model,verifier_model,strict_verifier_model,max_new_tokens_strict_verifier,pass1_accuracy,bonmav_accuracy,pass8_accuracy,total_exec_time_minutes
1,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,16,0.400,0.554,0.628,155.166667
3,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,deepseek-r1-distill-qwen-1.5b,2048,0.372,0.484,0.630,172.066667
5,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-1b-it,16,0.386,0.450,0.640,136.266667


In [55]:
selected_columns_4 = [
    'model',
    'verifier_model',
    'strict_verifier_model',
    'max_new_tokens_verifier',
    'max_new_tokens_strict_verifier',
    'pass1_accuracy',
    'bonmav_accuracy',
    'pass8_accuracy',
    'total_exec_time_minutes'
]
diff_verifier_max_tokens_filtered_df = rows[selected_columns_4].iloc[[0,1,6,7]]
diff_verifier_max_tokens_filtered_df



,model,verifier_model,strict_verifier_model,max_new_tokens_verifier,max_new_tokens_strict_verifier,pass1_accuracy,bonmav_accuracy,pass8_accuracy,total_exec_time_minutes
0,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,32768,16,0.428,0.572,0.636,298.466667
1,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,8192,16,0.400,0.554,0.628,155.166667
6,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,4096,16,0.396,0.552,0.634,103.866667
7,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,2048,16,0.394,0.536,0.618,71.316667


In [66]:
selected_columns_5 = [
    'model',
    'verifier_model',
    'strict_verifier_model',
    'max_new_tokens_model',
    'max_new_tokens_verifier',
    'pass1_accuracy',
    'bonmav_accuracy',
    'pass4_accuracy',
    'total_exec_time_minutes'
]
num_gen_4_rows = rows[(rows['num_generations'] == 4)][selected_columns_5]
custom_order = [8, 12, 14, 9, 13, 11]
reordered_df = num_gen_4_rows.loc[custom_order]
reordered_df

,model,verifier_model,strict_verifier_model,max_new_tokens_model,max_new_tokens_verifier,pass1_accuracy,bonmav_accuracy,pass4_accuracy,total_exec_time_minutes
8,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,2048,4096,0.398,0.510,0.560,59.716667
12,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,2048,2048,0.416,0.512,0.564,44.066667
14,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,2048,1024,0.364,0.486,0.554,34.166667
9,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,1024,2048,0.374,0.460,0.510,39.516667
13,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,1024,2048,0.358,0.452,0.502,45.683333
11,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,1024,1024,0.322,0.460,0.508,31.833333


In [57]:
.iloc[[8,12,14,9,11,13]]
a = ['model', 'verifier_model', 'strict_verifier_model',
       'num_generations', 'max_new_tokens_model', 'max_new_tokens_verifier',
       'max_new_tokens_strict_verifier',
       'bonmav_correct_count', 'bonmav_total_eval', 'bonmav_accuracy',
       'pass8_correct_count', 'pass8_total_eval', 'pass8_accuracy',
       'pass1_correct_count', 'pass1_total_eval', 'pass1_accuracy',
       'total_exec_time_minutes', 'pass4_correct_count', 'pass4_total_eval',
       'pass4_accuracy']

In [58]:
gemma1b_verified_by_deepseek = rows[(rows['model'] == 'gemma-3-1b-it') & (rows['verifier_model'] == 'deepseek-r1-distill-qwen-1.5b')][a]
gemma1b_verified_by_deepseek

,model,verifier_model,strict_verifier_model,num_generations,max_new_tokens_model,max_new_tokens_verifier,max_new_tokens_strict_verifier,bonmav_correct_count,bonmav_total_eval,bonmav_accuracy,pass8_correct_count,pass8_total_eval,pass8_accuracy,pass1_correct_count,pass1_total_eval,pass1_accuracy,total_exec_time_minutes,pass4_correct_count,pass4_total_eval,pass4_accuracy
0,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,8,2048,32768,16,286,500,0.572,318.0,500.0,0.636,214,500,0.428,298.466667,NaN,NaN,NaN
1,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,8,2048,8192,16,277,500,0.554,314.0,500.0,0.628,200,500,0.400,155.166667,NaN,NaN,NaN
3,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,deepseek-r1-distill-qwen-1.5b,8,2048,8192,2048,242,500,0.484,315.0,500.0,0.630,186,500,0.372,172.066667,NaN,NaN,NaN
5,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-1b-it,8,2048,8192,16,225,500,0.450,320.0,500.0,0.640,193,500,0.386,136.266667,NaN,NaN,NaN
6,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,8,2048,4096,16,276,500,0.552,317.0,500.0,0.634,198,500,0.396,103.866667,NaN,NaN,NaN
7,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,8,2048,2048,16,268,500,0.536,309.0,500.0,0.618,197,500,0.394,71.316667,NaN,NaN,NaN
8,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,4,2048,4096,16,255,500,0.510,NaN,NaN,NaN,199,500,0.398,59.716667,280.0,500.0,0.560
9,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,4,1024,2048,16,230,500,0.460,NaN,NaN,NaN,187,500,0.374,39.516667,255.0,500.0,0.510
11,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,4,1024,1024,16,230,500,0.460,NaN,NaN,NaN,161,500,0.322,31.833333,254.0,500.0,0.508
12,gemma-3-1b-it,deepseek-r1-distill-qwen-1.5b,gemma-3-4b-it,4,2048,2048,16,256,500,0.512,NaN,NaN,NaN,208,500,0.416,44.066667,282.0,500.0,0.564


In [59]:
rows.columns

Index(['model', 'verifier_model', 'strict_verifier_model', 'ntasks', 'dataset',
       'num_generations', 'max_new_tokens_model', 'max_new_tokens_verifier',
       'max_new_tokens_strict_verifier', 'slurm_id', 'path',
       'bonmav_correct_count', 'bonmav_total_eval', 'bonmav_accuracy',
       'pass8_correct_count', 'pass8_total_eval', 'pass8_accuracy',
       'pass1_correct_count', 'pass1_total_eval', 'pass1_accuracy',
       'total_exec_time_minutes', 'pass4_correct_count', 'pass4_total_eval',
       'pass4_accuracy'],
      dtype='object')